In [1]:
import astropy.table as at
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.coordinates as coord
from astropy.table import Table


from matplotlib.colors import LogNorm
from matplotlib.path import Path
from astropy.coordinates import SkyCoord, Distance

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# if you need these from your src package
from src.selection import *
from src.coords import *
from src.chemistry import *
from src.fitting import *
from src.plotting import *
from src.models import *

In [2]:
t = Table.read("astraAllStarASPCAP-0.8.fits.gz", format="fits", hdu=2)
t["sdss4_apogee_id"] = t["sdss4_apogee_id"].filled(-1)

dr17 = Table.read("allStarLite-dr17-synspec_rev1.fits", hdu=1)
t = at.join(
    t,
    dr17["APOGEE_ID", "SNR", "VHELIO_AVG"],
    keys_left="sdss4_apogee_id",
    keys_right="APOGEE_ID",
    join_type="left",
)
t.colnames

['sdss_id',
 'sdss4_apogee_id',
 'gaia_dr2_source_id',
 'gaia_dr3_source_id',
 'tic_v8_id',
 'healpix',
 'lead',
 'version_id',
 'catalogid',
 'catalogid21',
 'catalogid25',
 'catalogid31',
 'n_associated',
 'n_neighborhood',
 'sdss5_target_flags',
 'sdss4_apogee_target1_flags',
 'sdss4_apogee_target2_flags',
 'sdss4_apogee2_target1_flags',
 'sdss4_apogee2_target2_flags',
 'sdss4_apogee2_target3_flags',
 'sdss4_apogee_member_flags',
 'sdss4_apogee_extra_target_flags',
 'ra',
 'dec',
 'l',
 'b',
 'plx',
 'e_plx',
 'pmra',
 'e_pmra',
 'pmde',
 'e_pmde',
 'gaia_v_rad',
 'gaia_e_v_rad',
 'g_mag',
 'bp_mag',
 'rp_mag',
 'j_mag',
 'e_j_mag',
 'h_mag',
 'e_h_mag',
 'k_mag',
 'e_k_mag',
 'ph_qual',
 'bl_flg',
 'cc_flg',
 'w1_mag',
 'e_w1_mag',
 'w1_flux',
 'w1_dflux',
 'w1_frac',
 'w2_mag',
 'e_w2_mag',
 'w2_flux',
 'w2_dflux',
 'w2_frac',
 'w1uflags',
 'w2uflags',
 'w1aflags',
 'w2aflags',
 'mag4_5',
 'd4_5m',
 'rms_f4_5',
 'sqf_4_5',
 'mf4_5',
 'csf',
 'zgr_teff',
 'zgr_e_teff',
 'zgr_logg',

In [6]:
X_star = x_final
Y_star = y_final
Z_star = z_final
R_star = np.sqrt(X_star**2 + Y_star**2)
phi_star = np.arctan2(Y_star, X_star)
mgfe_star = mgfe_final

NameError: name 'x_final' is not defined

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture


# =========================================================
# SETTINGS
# =========================================================
SEARCH_MIN = -1.0
SEARCH_MAX = 1.0
EDGE_BUFFER_SEARCH = 0.10

MINN_SEG_TOTAL = 20
N_COMPONENTS = 2
RANDOM_STATE = 0
REG_COVAR = 1e-4

MIN_COMPONENT_WEIGHT = 0.15   # reject tiny spurious components
MAX_SIGMA_Z = 0.8             # reject very broad central components
ZLIM_HEATMAP = 0.6


# =========================================================
# FIT 1D GMM TO Z AND PICK CENTRAL COMPONENT
# =========================================================
def fit_gmm_z_midplane(
    Zs,
    n_components=2,
    random_state=0,
    reg_covar=1e-4,
    search_min=-1.0,
    search_max=1.0,
    min_component_weight=0.15,
    max_sigma_z=0.8,
):
    """
    Fit a 1D GaussianMixture to Z only.

    Selection rule:
    - choose the component whose mean is closest to Z=0
    - reject if the chosen component is outside the trusted window,
      too close to the trusted-window edge, too low-weight, or too broad
    """
    Zs = np.asarray(Zs, float)
    Zs = Zs[np.isfinite(Zs)]

    if len(Zs) < max(20, 8 * n_components):
        return {
            "status": "too_few_stars",
            "z_mid": np.nan,
            "gmm": None,
            "chosen_idx": None,
        }

    X = Zs[:, None]   # shape (n_samples, 1)

    try:
        gmm = GaussianMixture(
            n_components=n_components,
            random_state=random_state,
            reg_covar=reg_covar,
        ).fit(X)
    except Exception:
        return {
            "status": "fit_failed",
            "z_mid": np.nan,
            "gmm": None,
            "chosen_idx": None,
        }

    means = gmm.means_.ravel()
    weights = gmm.weights_.ravel()

    covs = np.asarray(gmm.covariances_)
    # covariance_type='full' by default, so in 1D this is usually (k,1,1)
    if covs.ndim == 3:
        sigma_z = np.sqrt(covs[:, 0, 0])
    elif covs.ndim == 2:
        sigma_z = np.sqrt(covs[:, 0])
    else:
        sigma_z = np.sqrt(covs)

    # choose the component closest to Z=0
    chosen_idx = int(np.argmin(np.abs(means)))
    z_mid = float(means[chosen_idx])

    # quality checks
    if not (search_min <= z_mid <= search_max):
        status = "chosen_component_outside_search_window"
        z_mid = np.nan
    elif (z_mid <= search_min + EDGE_BUFFER_SEARCH) or (z_mid >= search_max - EDGE_BUFFER_SEARCH):
        status = "chosen_component_near_edge"
        z_mid = np.nan
    elif weights[chosen_idx] < min_component_weight:
        status = "chosen_component_low_weight"
        z_mid = np.nan
    elif sigma_z[chosen_idx] > max_sigma_z:
        status = "chosen_component_too_broad"
        z_mid = np.nan
    else:
        status = "ok"

    return {
        "status": status,
        "z_mid": z_mid,
        "gmm": gmm,
        "chosen_idx": chosen_idx,
        "means": means,
        "weights": weights,
        "sigma_z": sigma_z,
    }


# =========================================================
# TEST ONE ANNULUS + SEGMENT
# =========================================================
Rmin_focus, Rmax_focus = 10.5, 11.0
seg_id = 4

mR = (R_star >= Rmin_focus) & (R_star < Rmax_focus)
mseg = mR & segm[seg_id]

Zs = np.asarray(Z_star[mseg], float)
Zs = Zs[np.isfinite(Zs)]

res_gmm_z = fit_gmm_z_midplane(
    Zs,
    n_components=N_COMPONENTS,
    random_state=RANDOM_STATE,
    reg_covar=REG_COVAR,
    search_min=SEARCH_MIN,
    search_max=SEARCH_MAX,
    min_component_weight=MIN_COMPONENT_WEIGHT,
    max_sigma_z=MAX_SIGMA_Z,
)

print("status:", res_gmm_z["status"])
print("z_mid:", res_gmm_z["z_mid"])

# plot histogram + component means
plt.figure(figsize=(7, 4.5))
plt.hist(Zs, bins=50, density=True, alpha=0.35, color="gray", label="Z histogram")

if res_gmm_z["gmm"] is not None:
    means = res_gmm_z["means"]
    weights = res_gmm_z["weights"]
    sigma_z = res_gmm_z["sigma_z"]
    chosen_idx = res_gmm_z["chosen_idx"]

    zgrid = np.linspace(np.nanmin(Zs), np.nanmax(Zs), 500)
    logprob = res_gmm_z["gmm"].score_samples(zgrid[:, None])
    pdf = np.exp(logprob)
    plt.plot(zgrid, pdf, color="black", lw=2, label="GMM total")

    # component curves
    for k in range(len(means)):
        mu = means[k]
        sig = sigma_z[k]
        w = weights[k]
        comp = w * (1.0 / (np.sqrt(2*np.pi) * sig)) * np.exp(-0.5 * ((zgrid - mu) / sig)**2)

        color = "red" if k == chosen_idx else "tab:blue"
        plt.plot(zgrid, comp, color=color, ls="--", lw=1.5)
        plt.axvline(mu, color=color, ls=":", alpha=0.7)

if np.isfinite(res_gmm_z["z_mid"]):
    plt.axvline(res_gmm_z["z_mid"], color="black", ls="--", lw=1.5, label=f"chosen z_mid={res_gmm_z['z_mid']:+.3f}")

plt.axvline(0, color="0.7", lw=1)
plt.xlim(-3, 3)
plt.xlabel("Z [kpc]")
plt.ylabel("Density")
plt.title(f"1D GMM on Z: annulus {Rmin_focus:.1f}-{Rmax_focus:.1f}, seg {seg_id}")
plt.legend()
plt.tight_layout()
plt.show()


# =========================================================
# RUN OVER ALL ANNULI + SEGMENTS
# =========================================================
def compute_df_gmm_z_midplane(
    R_star,
    Z_star,
    segm,
    seg_plot,
    R_edges=None,
    n_components=2,
    minN_seg_total=20,
    random_state=0,
    reg_covar=1e-4,
    search_min=-1.0,
    search_max=1.0,
    min_component_weight=0.15,
    max_sigma_z=0.8,
):
    if R_edges is None:
        R_edges = np.arange(0.5, 15.5 + 0.5, 0.5)

    rows = []

    for ir in range(len(R_edges) - 1):
        Rmin_focus = float(R_edges[ir])
        Rmax_focus = float(R_edges[ir + 1])

        mR = (R_star >= Rmin_focus) & (R_star < Rmax_focus)

        for seg_id in seg_plot:
            mseg = mR & segm[seg_id]
            good = mseg & np.isfinite(Z_star)

            Zs = np.asarray(Z_star[good], float)
            Nseg = len(Zs)

            if Nseg < minN_seg_total:
                rows.append(dict(
                    Rmin=Rmin_focus,
                    Rmax=Rmax_focus,
                    seg=int(seg_id),
                    Nseg=Nseg,
                    z_mid=np.nan,
                    status="too_few_stars",
                ))
                continue

            res = fit_gmm_z_midplane(
                Zs,
                n_components=n_components,
                random_state=random_state,
                reg_covar=reg_covar,
                search_min=search_min,
                search_max=search_max,
                min_component_weight=min_component_weight,
                max_sigma_z=max_sigma_z,
            )

            rows.append(dict(
                Rmin=Rmin_focus,
                Rmax=Rmax_focus,
                seg=int(seg_id),
                Nseg=Nseg,
                z_mid=res["z_mid"],
                status=res["status"],
            ))

    return pd.DataFrame(rows)


df_gmm_z = compute_df_gmm_z_midplane(
    R_star=R_star,
    Z_star=Z_star,
    segm=segm,
    seg_plot=SEG_PLOT,
    R_edges=None,
    n_components=N_COMPONENTS,
    minN_seg_total=MINN_SEG_TOTAL,
    random_state=RANDOM_STATE,
    reg_covar=REG_COVAR,
    search_min=SEARCH_MIN,
    search_max=SEARCH_MAX,
    min_component_weight=MIN_COMPONENT_WEIGHT,
    max_sigma_z=MAX_SIGMA_Z,
)

print(df_gmm_z["status"].value_counts(dropna=False))
display(df_gmm_z.head())


# =========================================================
# BUILD POLAR MAP
# =========================================================
def build_gmm_z_polar_map(
    df_gmm_z,
    crowd_bounds,
    seg_plot,
    seg_rest_id=None,
    zlim=0.6,
):
    if seg_rest_id is not None:
        seg_plot_heat = [s for s in seg_plot if s != seg_rest_id]
    else:
        seg_plot_heat = list(seg_plot)

    n_th_from_bounds = len(crowd_bounds) - 1
    if len(seg_plot_heat) != n_th_from_bounds:
        seg_plot_heat = list(range(1, n_th_from_bounds + 1))

    theta_edges = np.unwrap(np.array(crowd_bounds, float))
    for i in range(1, len(theta_edges)):
        while theta_edges[i] <= theta_edges[i - 1]:
            theta_edges[i] += 2 * np.pi

    R_edges_kept = np.array(
        sorted(set(df_gmm_z["Rmin"].tolist() + df_gmm_z["Rmax"].tolist())),
        float
    )

    n_r = len(R_edges_kept) - 1
    n_th = len(seg_plot_heat)

    Zmap_raw = np.full((n_r, n_th), np.nan, float)
    seg_to_col = {seg: j for j, seg in enumerate(seg_plot_heat)}

    good_rows = (df_gmm_z["status"] == "ok")

    for _, row in df_gmm_z.loc[good_rows].iterrows():
        seg = int(row["seg"])
        if seg not in seg_to_col:
            continue

        zmid = float(row["z_mid"])
        if not np.isfinite(zmid):
            continue

        Rmin = float(row["Rmin"])
        ir = np.searchsorted(R_edges_kept, Rmin, side="right") - 1
        if not (0 <= ir < n_r):
            continue

        it = seg_to_col[seg]
        Zmap_raw[ir, it] = zmid

    Zmap = np.where(
        np.isfinite(Zmap_raw) & (np.abs(Zmap_raw) <= zlim),
        Zmap_raw,
        np.nan
    )

    return Zmap, theta_edges, R_edges_kept, seg_plot_heat


Zmap_gmm_z, theta_edges_gmm_z, R_edges_gmm_z, seg_plot_heat_gmm_z = build_gmm_z_polar_map(
    df_gmm_z=df_gmm_z,
    crowd_bounds=crowd_bounds,
    seg_plot=SEG_PLOT,
    seg_rest_id=SEG_REST_ID if "SEG_REST_ID" in globals() else None,
    zlim=ZLIM_HEATMAP,
)

fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(8, 7))

pcm = ax.pcolormesh(
    theta_edges_gmm_z,
    R_edges_gmm_z,
    Zmap_gmm_z,
    shading="auto",
    cmap="RdBu_r",
    vmin=-ZLIM_HEATMAP,
    vmax= ZLIM_HEATMAP,
)

cbar = plt.colorbar(pcm, ax=ax, pad=0.12)
cbar.set_label(r"1D GMM midplane $z_{\rm mid}$ [kpc]")

ax.set_title("Midplane from 1D GaussianMixture on Z", pad=20)
plt.tight_layout()
plt.show()

NameError: name 'R_star' is not defined